In [ ]:
!pip install coqui-tts huggingface_hub

In [ ]:
from huggingface_hub import snapshot_download

# Download the model files from Hugging Face
model_dir = snapshot_download(repo_id="SYSPIN/vits_Maithili_Female")

print(f"Model downloaded to: {model_dir}")

In [ ]:
import os
import torch
import transformers.pytorch_utils as pt_utils

# 1. The Monkey-Patch: Trick Coqui into using standard PyTorch 'isin'
if not hasattr(pt_utils, 'isin_mps_friendly'):
    pt_utils.isin_mps_friendly = torch.isin

# 2. NOW we can safely import Coqui TTS without it crashing!
from TTS.api import TTS
from huggingface_hub import snapshot_download

# 3. Download the SYSPIN model
model_dir = snapshot_download(repo_id="SYSPIN/vits_Maithili_Female")
print(f"Model downloaded to: {model_dir}")

# 4. Initialize the TTS engine
tts = TTS(
    model_path=os.path.join(model_dir, "best_model.pth"),
    config_path=os.path.join(model_dir, "config.json"),
    progress_bar=False,
    gpu=True
)

# 5. Generate and Save Audio
maithili_text = "हमर गाममे आमक गाछ पर बहुत रास मञ्जरि आयल अछि।"
output_path = "syspin_maithili_output.wav"
tts.tts_to_file(text=maithili_text, file_path=output_path)

print(f"SYSPIN audio successfully generated at {output_path}")

In [ ]:
!pip install jiwer

In [ ]:
import json
import os
import shutil
import torch
import torchaudio
import scipy.io.wavfile
from google.colab import files

# ==========================================
# 0. FIX DEPENDENCY CRASHES (Coqui)
# ==========================================
# Monkey-patch PyTorch to fix the Coqui-TTS import crash
import transformers.pytorch_utils as pt_utils
if not hasattr(pt_utils, 'isin_mps_friendly'):
    pt_utils.isin_mps_friendly = torch.isin

# Safely import the remaining libraries
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from TTS.api import TTS
from huggingface_hub import snapshot_download
from jiwer import wer, cer

# ==========================================
# 1. SETUP & DIRECTORIES
# ==========================================
device = "cuda:0" if torch.cuda.is_available() else "cpu"
output_dir = "syspin_maithili_outputs"
os.makedirs(output_dir, exist_ok=True)

# Safely parse the JSON file
with open('maithili_evaluation_set.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

if isinstance(raw_data, dict):
    for key, value in raw_data.items():
        if isinstance(value, list):
            raw_data = value
            break

sentences = []
for item in raw_data:
    if isinstance(item, dict):
        # Look for standard keys first
        if 'text' in item and isinstance(item['text'], str):
            sentences.append(item['text'])
        elif 'sentence' in item and isinstance(item['sentence'], str):
            sentences.append(item['sentence'])
        else:
            # Fallback: aggressively hunt for the first string value it can find
            for k, v in item.items():
                if isinstance(v, str):
                    sentences.append(v)
                    break
    elif isinstance(item, str):
        sentences.append(item)

print(f"Loaded {len(sentences)} sentences for evaluation.")
# Print the first sentence to absolutely verify we grabbed text, not an ID!
if len(sentences) > 0:
    print(f"Sample Text: {sentences[0][:50]}...\n")

# ==========================================
# 2. LOAD MODELS
# ==========================================
print("Loading SYSPIN VITS Model via Coqui...")
syspin_dir = snapshot_download(repo_id="SYSPIN/vits_Maithili_Female")
syspin_tts = TTS(
    model_path=os.path.join(syspin_dir, "best_model.pth"),
    config_path=os.path.join(syspin_dir, "config.json"),
    progress_bar=False,
    gpu=(device == "cuda:0")
)

print("Loading Whisper Medium ASR Model...")
asr_id = "openai/whisper-medium"
asr_processor = WhisperProcessor.from_pretrained(asr_id)
asr_model = WhisperForConditionalGeneration.from_pretrained(asr_id).to(device)

# Force Whisper to transcribe to Hindi (Devanagari)
forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language="hindi", task="transcribe")

# ==========================================
# 3. HELPER FUNCTION
# ==========================================
def transcribe_audio_whisper(audio_path):
    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)

    input_features = asr_processor(
        waveform.squeeze().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = asr_model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

    return asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

# ==========================================
# 4. GENERATION & EVALUATION LOOP
# ==========================================
results = []

for i, text in enumerate(sentences):
    print(f"Processing Sentence {i+1}/{len(sentences)}...")

    syspin_path = os.path.join(output_dir, f"syspin_sent_{i+1}.wav")

    # --- Generate SYSPIN ---
    # Coqui handles the Devanagari text processing automatically
    syspin_tts.tts_to_file(text=text, file_path=syspin_path, split_sentences=False)

    # --- Transcribe & Evaluate ---
    syspin_transcription = transcribe_audio_whisper(syspin_path)
    syspin_wer = wer(text, syspin_transcription)
    syspin_cer = cer(text, syspin_transcription)

    # --- Store Metrics ---
    results.append({
        "id": i + 1,
        "original_text": text,
        "syspin_transcription": syspin_transcription,
        "syspin_wer": syspin_wer,
        "syspin_cer": syspin_cer
    })

# ==========================================
# 5. SAVE METRICS & ZIP ARCHIVE
# ==========================================
with open(os.path.join(output_dir, "evaluation_metrics.json"), 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("\nEvaluation Complete. Zipping files...")
zip_filename = "syspin_maithili_evaluation"
shutil.make_archive(zip_filename, 'zip', output_dir)

print("Triggering download...")
files.download(f"{zip_filename}.zip")

In [ ]:
import json
import os
import shutil
import torch
import torchaudio
from google.colab import files

# ==========================================
# 0. FIX DEPENDENCY CRASHES
# ==========================================
# Monkey-patch PyTorch to fix the Coqui-TTS import crash
import transformers.pytorch_utils as pt_utils
if not hasattr(pt_utils, 'isin_mps_friendly'):
    pt_utils.isin_mps_friendly = torch.isin

# Safely import remaining libraries
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from TTS.api import TTS
from huggingface_hub import snapshot_download
from jiwer import wer, cer

# ==========================================
# 1. SETUP & DIRECTORIES
# ==========================================
device = "cuda:0" if torch.cuda.is_available() else "cpu"
output_dir = "syspin_maithili_outputs"
os.makedirs(output_dir, exist_ok=True)

# Safely parse the JSON file, aggressively hunting for text strings
with open('maithili_evaluation_set.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

if isinstance(raw_data, dict):
    for key, value in raw_data.items():
        if isinstance(value, list):
            raw_data = value
            break

sentences = []
for item in raw_data:
    if isinstance(item, dict):
        if 'text' in item and isinstance(item['text'], str):
            sentences.append(item['text'])
        elif 'sentence' in item and isinstance(item['sentence'], str):
            sentences.append(item['sentence'])
        else:
            for k, v in item.items():
                if isinstance(v, str):
                    sentences.append(v)
                    break
    elif isinstance(item, str):
        sentences.append(item)

print(f"Loaded {len(sentences)} sentences for evaluation.\n")

# ==========================================
# 2. LOAD MODELS
# ==========================================
print("Loading SYSPIN VITS Model via Coqui...")
syspin_dir = snapshot_download(repo_id="SYSPIN/vits_Maithili_Female")
syspin_tts = TTS(
    model_path=os.path.join(syspin_dir, "best_model.pth"),
    config_path=os.path.join(syspin_dir, "config.json"),
    progress_bar=False,
    gpu=(device == "cuda:0")
)

print("Loading Whisper Medium ASR Model...")
asr_id = "openai/whisper-medium"
asr_processor = WhisperProcessor.from_pretrained(asr_id)
asr_model = WhisperForConditionalGeneration.from_pretrained(asr_id).to(device)

# Force Whisper to transcribe to Devanagari script using the Hindi token
forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language="hindi", task="transcribe")

# ==========================================
# 3. HELPER FUNCTION
# ==========================================
def transcribe_audio_whisper(audio_path):
    waveform, sr = torchaudio.load(audio_path)
    # Ensure 16kHz sample rate for Whisper
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)

    input_features = asr_processor(
        waveform.squeeze().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = asr_model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

    return asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

# ==========================================
# 4. GENERATION & EVALUATION LOOP
# ==========================================
results = []

for i, text in enumerate(sentences):
    print(f"Processing Sentence {i+1}/{len(sentences)}...")

    syspin_path = os.path.join(output_dir, f"syspin_sent_{i+1}.wav")

    # --- Generate SYSPIN ---
    # split_sentences=False strictly forces the text through without passing it to pysbd
    syspin_tts.tts_to_file(text=text, file_path=syspin_path, split_sentences=False)

    # --- Transcribe & Evaluate ---
    syspin_transcription = transcribe_audio_whisper(syspin_path)
    syspin_wer = wer(text, syspin_transcription)
    syspin_cer = cer(text, syspin_transcription)

    # --- Store Metrics ---
    results.append({
        "id": i + 1,
        "original_text": text,
        "syspin_transcription": syspin_transcription,
        "syspin_wer": syspin_wer,
        "syspin_cer": syspin_cer
    })

# ==========================================
# 5. SAVE METRICS & ZIP ARCHIVE
# ==========================================
with open(os.path.join(output_dir, "evaluation_metrics.json"), 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("\nEvaluation Complete. Zipping files...")
zip_filename = "syspin_maithili_evaluation"
shutil.make_archive(zip_filename, 'zip', output_dir)

print("Triggering download...")
files.download(f"{zip_filename}.zip")